# 🏭 Üretim Anomali Tespiti — One-Class SVM (OCSVM)
## Benchmarking: IF ve XGBoost ile Kıyaslamalı Analiz
> **Veri:** `excel_partiler_total.xlsx` &nbsp;|&nbsp; **7 Sensör (S1–S7)** &nbsp;|&nbsp; **12 Parti (6N + 6A)**  
> **Yaklaşım:** Normal partilerle eğitilmiş RBF-kernel OCSVM → `-decision_function` → Anomali Skoru

### Notebook Akışı
1. Kurulum & İçe Aktarma
2. Veri Yükleme & Temizleme
3–5. EDA (Genel Bakış · Zaman Serileri · KDE)
6. Sliding Window Feature Extraction
7. Eğitim / Test Ayrımı & **StandardScaler** (OCSVM için kritik)
8. Model Eğitimi (OneClassSVM — RBF kernel)
9. Anomali Skoru (`-decision_function`)
10. Eşik Optimizasyonu (F1 max)
11. Performans Metrikleri (ROC · PR · CM)
12. Anomali Skor Zaman Serileri
13. Sensör Katkı Analizi (Permütasyon Proxy)
14. Özet Tablo & Dosya İndirme

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   ÜRETİM ANOMALİ TESPİTİ — One-Class SVM (OCSVM)                       ║
# ║   Benchmarking: IF ve XGBoost ile kıyaslamalı analiz                    ║
# ║   Veri: excel_partiler_total.xlsx  |  12 Parti  |  7 Sensör             ║
# ╚══════════════════════════════════════════════════════════════════════════╝


### HÜCRE 1: Kurulum & İçe Aktarma

In [ ]:
# ── HÜCRE 1: Kurulum & İçe Aktarma ─────────────────────────────────────────

!pip install -q openpyxl scikit-learn matplotlib seaborn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_recall_curve, roc_curve, average_precision_score,
)
import os, json
from google.colab import files

# ── Görsel stil (IF ve XGBoost ile birebir aynı) ──
plt.rcParams.update({
    "figure.facecolor": "#0F1117",
    "axes.facecolor":   "#1A1D27",
    "axes.edgecolor":   "#3A3F54",
    "axes.labelcolor":  "#C8CDD8",
    "xtick.color":      "#8891A5",
    "ytick.color":      "#8891A5",
    "text.color":       "#E2E5EE",
    "grid.color":       "#2A2F42",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
    "axes.titlesize":   11,
    "axes.titleweight": "bold",
    "axes.titlecolor":  "#FFFFFF",
    "figure.dpi":       130,
})

PALETTE_NORMAL = "#3DD68C"
PALETTE_ANOM   = "#FF5370"
PALETTE_SCORE  = "#7C83FD"
PALETTE_THRESH = "#FFB547"

print("✓ Kurulum ve içe aktarma tamamlandı.")


### HÜCRE 2: Veri Yükleme & Temizleme

In [ ]:
# ── HÜCRE 2: Veri Yükleme & Temizleme ──────────────────────────────────────

# ── DOSYA YÜKLEME ────────────────────────────────────────────────────────────
# Aşağıdaki hücre dosya seçiciyi açar; excel_partiler_total.xlsx seçin
from google.colab import files as _colab_files
import io as _io, os as _os, pandas as pd

_uploaded = _colab_files.upload()
FILENAME  = list(_uploaded.keys())[0]

# Colab farklı versiyonlarda uploaded[key] olarak bytes ya da path döndürür.
# Her iki durumu da kapsayan savunmacı okuma:
def _safe_read_excel(uploaded_dict, filename, sheet=0):
    val = uploaded_dict[filename]

    # 1) Zaten bytes ise doğruca BytesIO'ya ver
    if isinstance(val, (bytes, bytearray)):
        raw = bytes(val)
        if raw[:4] == b'PK\x03\x04':          # geçerli xlsx zip imzası
            return pd.read_excel(_io.BytesIO(raw), sheet_name=sheet,
                                 engine="openpyxl")

    # 2) Colab bazen dosyayı /content/ altına yazar, oradan oku
    for candidate in [
        f"/content/{filename}",
        f"/content/{filename.split('(')[0].strip()}",   # "(1)" suffix temizle
        filename,                                          # mutlak yol geldiyse
    ]:
        if _os.path.isfile(candidate):
            with open(candidate, "rb") as _f:
                raw = _f.read()
            if raw[:4] == b'PK\x03\x04':
                return pd.read_excel(_io.BytesIO(raw), sheet_name=sheet,
                                     engine="openpyxl")

    # 3) Son çare: val'ı string path olarak dene
    if isinstance(val, str) and _os.path.isfile(val):
        with open(val, "rb") as _f:
            raw = _f.read()
        return pd.read_excel(_io.BytesIO(raw), sheet_name=sheet,
                             engine="openpyxl")

    raise RuntimeError(
        f"Excel dosyası okunamadı. Yüklenen anahtar: {filename}. "
        "Lütfen dosyayı tekrar yükleyin."
    )

df_raw = _safe_read_excel(_uploaded, FILENAME, sheet=0)
print(f"✓ Dosya okundu — shape: {df_raw.shape}")


# ── VERİ TEMİZLEME ──────────────────────────────────────────────────────────
_SENSOR_RAW = [
    "S1_AK_Sicakligi", "S2_BK1_Sicakligi", "S3_BK2_Sicaklik",
    "S4_AK_Seviyesi",  "S5_Iletkenlik",     "S6_BK1_Seviyesi",
    "S7_BK2_Seviyesi"
]
for _col in _SENSOR_RAW:
    if _col in df_raw.columns:
        df_raw[_col] = pd.to_numeric(df_raw[_col], errors="coerce")

df_raw[_SENSOR_RAW] = df_raw[_SENSOR_RAW].ffill().fillna(0)
print(f"✓ Veri temizlendi — kalan NaN: {df_raw[_SENSOR_RAW].isna().sum().sum()}")

# ── Sabitler ────────────────────────────────────────────────────────────────
SENSOR_COLS = [
    "S1_AK_Sicakligi", "S2_BK1_Sicakligi", "S3_BK2_Sicaklik",
    "S4_AK_Seviyesi",  "S5_Iletkenlik",     "S6_BK1_Seviyesi",
    "S7_BK2_Seviyesi"
]
SENSOR_LABELS = [
    "AK Sıcaklığı", "BK1 Sıcaklığı", "BK2 Sıcaklık",
    "AK Seviyesi",  "İletkenlik",     "BK1 Seviyesi",
    "BK2 Seviyesi"
]
SENSOR_COLS_NO_S4   = [s for s in SENSOR_COLS   if s != "S4_AK_Seviyesi"]
SENSOR_LABELS_NO_S4 = [l for l in SENSOR_LABELS if l != "AK Seviyesi"]

WINDOW_SIZE = 30   # IF ve XGBoost ile aynı

# Etiket: Kusur sütunu (0=normal, 1=anormal)
df_raw["label"] = df_raw["Kusur"].astype(int)

batch_labels = (
    df_raw.groupby("batch_id")["label"]
    .first()
    .reset_index()
)
batch_labels["durum"] = batch_labels["label"].map({0: "Normal", 1: "Anormal"})

print(f"✓ Veri yüklendi: {len(df_raw):,} satır | {df_raw['batch_id'].nunique()} parti")
print(f"  Normal  : {(batch_labels['label']==0).sum()} parti → eğitim")
print(f"  Anormal : {(batch_labels['label']==1).sum()} parti → test")
print()
print(batch_labels.to_string(index=False))


### HÜCRE 3: EDA — Genel Bakış (S4 ayrı subplot)

In [ ]:
# ── HÜCRE 3: EDA — Genel Bakış (S4 ayrı subplot) ────────────────────────────

fig = plt.figure(figsize=(20, 14))
fig.suptitle("EDA — Parti Genel Bakış  [OCSVM Notebook]",
             fontsize=14, color="#FFFFFF", y=1.01)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.32)

# 3a: Parti uzunlukları
ax1 = fig.add_subplot(gs[0, 0])
lengths = df_raw.groupby("batch_id")["t"].max().reset_index()
lengths = lengths.merge(batch_labels, on="batch_id")
colors_bar = [PALETTE_ANOM if l == 1 else PALETTE_NORMAL
              for l in lengths["label"]]
bars = ax1.barh(lengths["batch_id"].astype(str), lengths["t"],
                color=colors_bar, alpha=0.85, edgecolor="none")
ax1.axvline(lengths["t"].mean(), color=PALETTE_THRESH, ls="--", lw=1.2)
for bar, val in zip(bars, lengths["t"]):
    ax1.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             str(val), va="center", fontsize=7, color="#C8CDD8")
ax1.set_xlim(0, lengths["t"].max() * 1.18)
ax1.set_xlabel("Ölçüm Sayısı (dakika)")
ax1.set_title("Parti Uzunlukları")
ax1.legend(handles=[
    mpatches.Patch(color=PALETTE_NORMAL, label="Normal"),
    mpatches.Patch(color=PALETTE_ANOM,   label="Anormal"),
    Line2D([0],[0], color=PALETTE_THRESH, ls="--", lw=1.2,
           label=f"Ort. {lengths['t'].mean():.0f} dk"),
], fontsize=7)

# 3b: Boxplot — S4 HARİÇ
ax2 = fig.add_subplot(gs[0, 1])
melt_no_s4 = df_raw[SENSOR_COLS_NO_S4 + ["label"]].copy()
melt_no_s4.columns = SENSOR_LABELS_NO_S4 + ["label"]
melt_long_no_s4 = melt_no_s4.melt(id_vars="label", var_name="Sensör", value_name="Değer")
melt_long_no_s4["Durum"] = melt_long_no_s4["label"].map({0:"Normal",1:"Anormal"})
sns.boxplot(
    data=melt_long_no_s4, x="Sensör", y="Değer", hue="Durum",
    palette={"Normal": PALETTE_NORMAL, "Anormal": PALETTE_ANOM},
    ax=ax2, linewidth=0.7, fliersize=1.5, width=0.6,
    boxprops=dict(alpha=0.85),
)
ax2.set_title("Sensör Dağılımı: Normal vs Anormal  (S4 hariç)")
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=25)
ax2.legend(fontsize=7)

# 3c: Boxplot — Sadece S4
ax3 = fig.add_subplot(gs[1, 0])
melt_s4 = df_raw[["S4_AK_Seviyesi","label"]].copy()
melt_s4["Durum"] = melt_s4["label"].map({0:"Normal",1:"Anormal"})
sns.boxplot(
    data=melt_s4, x="Durum", y="S4_AK_Seviyesi", hue="Durum",
    palette={"Normal": PALETTE_NORMAL, "Anormal": PALETTE_ANOM},
    ax=ax3, linewidth=0.7, fliersize=1.5, width=0.45,
    boxprops=dict(alpha=0.85), legend=False,
)
ax3.set_title("AK Seviyesi (S4) — Ayrı Ölçek")
ax3.set_xlabel("")
ax3.set_ylabel("AK Seviyesi")

# 3d: Sensör ortalamaları — S4 HARİÇ
ax4 = fig.add_subplot(gs[1, 1])
normal_mean = df_raw[df_raw["label"]==0][SENSOR_COLS_NO_S4].mean()
anom_mean   = df_raw[df_raw["label"]==1][SENSOR_COLS_NO_S4].mean()
x = np.arange(len(SENSOR_LABELS_NO_S4))
w = 0.35
ax4.bar(x - w/2, normal_mean, w, label="Normal",
        color=PALETTE_NORMAL, alpha=0.85, edgecolor="none")
ax4.bar(x + w/2, anom_mean,   w, label="Anormal",
        color=PALETTE_ANOM,   alpha=0.85, edgecolor="none")
ax4.set_xticks(x)
ax4.set_xticklabels(SENSOR_LABELS_NO_S4, rotation=25, fontsize=8)
ax4.set_title("Sensör Ortalamaları: Normal vs Anormal  (S4 hariç)")
ax4.legend(fontsize=8)
ax4.set_ylabel("Ortalama Değer")

# 3e: AK Seviyesi ortalama — ayrı bar
ax5 = fig.add_subplot(gs[2, 0])
s4_means = {
    "Normal" : df_raw[df_raw["label"]==0]["S4_AK_Seviyesi"].mean(),
    "Anormal": df_raw[df_raw["label"]==1]["S4_AK_Seviyesi"].mean(),
}
ax5.bar(["Normal","Anormal"], list(s4_means.values()),
        color=[PALETTE_NORMAL, PALETTE_ANOM],
        alpha=0.85, edgecolor="none", width=0.4)
for i,(k,v) in enumerate(s4_means.items()):
    ax5.text(i, v+30, f"{v:.0f}", ha="center", fontsize=9, color="#E2E5EE")
ax5.set_title("AK Seviyesi (S4) — Ortalama  (Ayrı Ölçek)")
ax5.set_ylabel("Ortalama AK Seviyesi")

# 3f: Korelasyon matrisi (normal partiler)
ax6 = fig.add_subplot(gs[2, 1])
corr = df_raw[df_raw["label"]==0][SENSOR_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, ax=ax6, mask=mask, cmap=cmap, center=0,
            annot=True, fmt=".2f", annot_kws={"size": 6},
            linewidths=0.4, square=True,
            xticklabels=SENSOR_LABELS, yticklabels=SENSOR_LABELS,
            cbar_kws={"shrink": 0.7})
ax6.set_title("Sensör Korelasyonu (Normal Partiler)")
ax6.tick_params(axis="x", rotation=30, labelsize=6)
ax6.tick_params(axis="y", rotation=0,  labelsize=6)

plt.savefig("eda_genel_bakis_ocsvm.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_genel_bakis_ocsvm.png")
print("✓ EDA Genel Bakış indirildi.")


### HÜCRE 4: EDA — Sensör Zaman Serileri

In [ ]:
# ── HÜCRE 4: EDA — Sensör Zaman Serileri ────────────────────────────────────

fig, axes = plt.subplots(4, 2, figsize=(18, 16))
fig.suptitle("EDA — Sensör Zaman Serileri (Tüm Partiler)",
             fontsize=13, color="#FFFFFF")
axes = axes.flatten()

for i, (sensor, label_tr) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    ax = axes[i]
    for _, row in batch_labels.iterrows():
        bid   = row["batch_id"]
        lbl   = row["label"]
        bdata = df_raw[df_raw["batch_id"]==bid].sort_values("t")
        color = PALETTE_ANOM if lbl==1 else PALETTE_NORMAL
        ax.plot(bdata["t"], bdata[sensor],
                color=color,
                alpha=0.55 if lbl==1 else 0.70,
                lw=1.0   if lbl==1 else 0.8)
    ax.set_title(label_tr)
    ax.set_xlabel("t (ölçüm sırası)", fontsize=7)
    ax.set_ylabel("Değer", fontsize=7)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(handles=[
            Line2D([0],[0], color=PALETTE_NORMAL, lw=2, label="Normal"),
            Line2D([0],[0], color=PALETTE_ANOM,   lw=2, label="Anormal"),
        ], fontsize=8)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("eda_zaman_serileri_ocsvm.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_zaman_serileri_ocsvm.png")
print("✓ Zaman serileri grafiği indirildi.")


### HÜCRE 5: EDA — KDE Dağılımları & İstatistikler

In [ ]:
# ── HÜCRE 5: EDA — KDE Dağılımları & İstatistikler ──────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("EDA — Sensör Dağılımları (KDE)", fontsize=13, color="#FFFFFF")
axes = axes.flatten()

normal_data = df_raw[df_raw["label"]==0]
anom_data   = df_raw[df_raw["label"]==1]

for i, (sensor, label_tr) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    ax = axes[i]
    sns.kdeplot(normal_data[sensor].dropna(), ax=ax,
                color=PALETTE_NORMAL, fill=True, alpha=0.35, lw=1.5, label="Normal")
    sns.kdeplot(anom_data[sensor].dropna(),   ax=ax,
                color=PALETTE_ANOM,   fill=True, alpha=0.35, lw=1.5, label="Anormal")
    ax.set_title(label_tr, fontsize=9)
    ax.set_xlabel("Değer", fontsize=7)
    ax.set_ylabel("Yoğunluk", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("eda_kde_dagilimlar_ocsvm.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_kde_dagilimlar_ocsvm.png")
print("✓ KDE dağılım grafiği indirildi.")

print("\n── SENSÖR İSTATİSTİKLERİ (Normal) ──")
print(normal_data[SENSOR_COLS].describe().round(2).to_string())
print("\n── SENSÖR İSTATİSTİKLERİ (Anormal) ──")
print(anom_data[SENSOR_COLS].describe().round(2).to_string())


### HÜCRE 6: Sliding Window Feature Extraction

In [ ]:
# ── HÜCRE 6: Sliding Window Feature Extraction ──────────────────────────────
# IF ile birebir aynı: t anındaki pencere → flatten → 210 feature
# OCSVM için StandardScaler kritik — mesafe tabanlı model

def extract_window_features(df, window_size=WINDOW_SIZE):
    """
    Her zaman noktası için son window_size adımlık pencereyi flatten eder.
    Çıktı: (n_valid_points, window_size × n_sensors) feature matrisi + meta
    """
    results = []
    for batch_id, group in df.groupby("batch_id"):
        group   = group.sort_values("t").reset_index(drop=True)
        sensors = group[SENSOR_COLS].values   # (n, 7)
        label   = group["label"].iloc[0]
        n_rows  = len(sensors)

        for t_idx in range(window_size - 1, n_rows):
            window = sensors[t_idx - window_size + 1 : t_idx + 1]  # (30, 7)
            flat   = window.flatten()                               # (210,)
            results.append({
                "batch_id": batch_id,
                "t":        group["t"].iloc[t_idx],
                "label":    label,
                "features": flat,
            })

    feat_df = pd.DataFrame(results)
    X       = np.vstack(feat_df["features"].values)
    meta    = feat_df[["batch_id","t","label"]].reset_index(drop=True)

    n_feat = window_size * len(SENSOR_COLS)
    print(f"✓ Feature extraction tamamlandı")
    print(f"  Window size    : {window_size} dk")
    print(f"  Feature sayısı : {n_feat} ({window_size} × {len(SENSOR_COLS)})")
    print(f"  Toplam örnek   : {len(X):,}")
    return X, meta


X_all, meta_all = extract_window_features(df_raw)


### HÜCRE 7: Eğitim / Test Ayrımı & StandardScaler

In [ ]:
# ── HÜCRE 7: Eğitim / Test Ayrımı & StandardScaler ──────────────────────────
# OCSVM mesafe tabanlıdır → StandardScaler ZORUNLU

normal_batch_ids = batch_labels[batch_labels["label"]==0]["batch_id"].tolist()
anom_batch_ids   = batch_labels[batch_labels["label"]==1]["batch_id"].tolist()

train_mask = meta_all["batch_id"].isin(normal_batch_ids).values

X_train   = X_all[train_mask]
X_test    = X_all
meta_test = meta_all.copy()

# Ölçekleme: SADECE eğitim verisiyle fit et
scaler    = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"✓ Eğitim seti  : {len(X_train):,} örnek — {len(normal_batch_ids)} normal parti")
print(f"✓ Test seti    : {len(X_test):,}  örnek — 12 parti (6N + 6A)")
print(f"✓ StandardScaler uygulandı")
print(f"  Eğitim ortalama (ilk 3 feat): {scaler.mean_[:3].round(3)}")
print(f"  Eğitim std     (ilk 3 feat): {scaler.scale_[:3].round(3)}")


### HÜCRE 8: Model Eğitimi

In [ ]:
# ── HÜCRE 8: Model Eğitimi ───────────────────────────────────────────────────
# OCSVM hiperparametreleri:
#   kernel='rbf' : Doğrusal olmayan sınır için en iyi seçim
#   nu           : Eğitim hatası üst sınırı ≈ beklenen anomali oranı (0.05)
#   gamma='scale': 1/(n_features * X.var()) — otomatik ölçekleme

model = OneClassSVM(
    kernel = "rbf",
    nu     = 0.05,      # IF contamination ile eşdeğer
    gamma  = "scale",
)
model.fit(X_train_sc)

# Support vector sayısı — modelin ne kadar "sıkı" öğrendiğini gösterir
n_sv = model.support_vectors_.shape[0]
print(f"✓ OCSVM modeli eğitildi")
print(f"  Kernel          : RBF")
print(f"  nu              : 0.05")
print(f"  gamma           : scale")
print(f"  Support vector  : {n_sv:,} ({n_sv/len(X_train)*100:.1f}% eğitim verisi)")


### HÜCRE 9: Anomali Skoru Hesaplama

In [ ]:
# ── HÜCRE 9: Anomali Skoru Hesaplama ────────────────────────────────────────
# decision_function: pozitif = normal sınır içi, negatif = anomali
# Ters çevirerek: yüksek skor = yüksek anomali (IF ve XGBoost ile aynı yön)

raw_scores  = model.decision_function(X_test_sc)   # büyük = daha normal
anom_scores = -raw_scores                           # büyük = daha anomalik

meta_test = meta_test.copy()
meta_test["anomaly_score"] = anom_scores
# İkili tahmin (karar sınırı sıfır noktası)
meta_test["pred_binary_raw"] = np.where(
    model.predict(X_test_sc) == -1, 1, 0
)

print(f"✓ Anomali skoru hesaplandı (-decision_function)")
print(f"  Skor aralığı    : [{anom_scores.min():.4f}, {anom_scores.max():.4f}]")
print(f"  Normal ort.     : {anom_scores[meta_test['label']==0].mean():.4f}")
print(f"  Anormal ort.    : {anom_scores[meta_test['label']==1].mean():.4f}")
print(f"  Karar sınırı    : 0.0 (ham skor — eşik optimizasyonuyla rafine edilecek)")


### HÜCRE 10: Eşik Optimizasyonu

In [ ]:
# ── HÜCRE 10: Eşik Optimizasyonu ────────────────────────────────────────────
# IF ile birebir aynı mantık: parti düzeyi max skor → F1 maksimize eden eşik

def batch_level_eval(meta_df, threshold):
    grp = meta_df.groupby("batch_id").agg(
        true_label = ("label",         "first"),
        max_score  = ("anomaly_score", "max"),
        mean_score = ("anomaly_score", "mean"),
    ).reset_index()
    grp["pred_label"] = (grp["max_score"] > threshold).astype(int)
    return grp

thresholds  = np.percentile(anom_scores, np.arange(70, 99, 0.5))
thr_results = []
for thr in thresholds:
    grp = batch_level_eval(meta_test, thr)
    try:
        auc = roc_auc_score(grp["true_label"], grp["max_score"])
    except Exception:
        auc = 0.0
    tp = ((grp["pred_label"]==1) & (grp["true_label"]==1)).sum()
    fp = ((grp["pred_label"]==1) & (grp["true_label"]==0)).sum()
    fn = ((grp["pred_label"]==0) & (grp["true_label"]==1)).sum()
    f1 = tp / (tp + 0.5*(fp + fn) + 1e-9)
    thr_results.append({"threshold":thr,"f1":f1,"auc":auc,
                        "tp":tp,"fp":fp,"fn":fn})

thr_df   = pd.DataFrame(thr_results)
best_row = thr_df.loc[thr_df["f1"].idxmax()]
BEST_THRESHOLD = best_row["threshold"]

print(f"✓ Eşik optimizasyonu tamamlandı")
print(f"  En iyi eşik : {BEST_THRESHOLD:.4f}")
print(f"  F1 skoru    : {best_row['f1']:.3f}")
print(f"  ROC-AUC     : {best_row['auc']:.3f}")
print(f"  TP: {int(best_row['tp'])}  FP: {int(best_row['fp'])}  FN: {int(best_row['fn'])}")


### HÜCRE 11: Performans Metrikleri & Görselleştirme

In [ ]:
# ── HÜCRE 11: Performans Metrikleri & Görselleştirme ────────────────────────

batch_results = batch_level_eval(meta_test, BEST_THRESHOLD)

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Performans Metrikleri — One-Class SVM",
             fontsize=14, color="#FFFFFF", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 11a: Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(batch_results["true_label"], batch_results["pred_label"])
sns.heatmap(cm, annot=True, fmt="d", ax=ax1, cmap="YlOrRd",
            xticklabels=["Normal","Anormal"],
            yticklabels=["Normal","Anormal"],
            annot_kws={"size":16,"weight":"bold"},
            linewidths=0.5, cbar=False)
ax1.set_title("Confusion Matrix (Parti Düzeyi)")
ax1.set_xlabel("Tahmin", fontsize=9)
ax1.set_ylabel("Gerçek",  fontsize=9)

# 11b: ROC Eğrisi
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(batch_results["true_label"], batch_results["max_score"])
auc_val = roc_auc_score(batch_results["true_label"], batch_results["max_score"])
ax2.plot(fpr, tpr, color=PALETTE_SCORE, lw=2, label=f"ROC AUC = {auc_val:.3f}")
ax2.plot([0,1],[0,1], "w--", lw=0.8, alpha=0.5)
ax2.fill_between(fpr, tpr, alpha=0.15, color=PALETTE_SCORE)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Eğrisi (Parti Düzeyi)")
ax2.legend(fontsize=9)
ax2.set_xlim([-0.02, 1.02]); ax2.set_ylim([-0.02, 1.05])

# 11c: Precision-Recall
ax3 = fig.add_subplot(gs[0, 2])
prec, rec, _ = precision_recall_curve(
    batch_results["true_label"], batch_results["max_score"])
ap = average_precision_score(batch_results["true_label"], batch_results["max_score"])
ax3.plot(rec, prec, color=PALETTE_ANOM, lw=2, label=f"AP = {ap:.3f}")
ax3.fill_between(rec, prec, alpha=0.15, color=PALETTE_ANOM)
ax3.set_xlabel("Recall"); ax3.set_ylabel("Precision")
ax3.set_title("Precision-Recall Eğrisi")
ax3.legend(fontsize=9)
ax3.set_xlim([-0.02, 1.02]); ax3.set_ylim([-0.02, 1.05])

# 11d: Eşik vs F1
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(thr_df["threshold"], thr_df["f1"], color=PALETTE_SCORE, lw=1.8)
ax4.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5,
            label=f"Best = {BEST_THRESHOLD:.4f}")
ax4.set_xlabel("Eşik Değeri (-decision_function)")
ax4.set_ylabel("F1 Skoru")
ax4.set_title("Eşik Optimizasyonu")
ax4.legend(fontsize=8)

# 11e: Parti bazında max anomali skoru
ax5 = fig.add_subplot(gs[1, 1])
br = batch_results.sort_values("max_score", ascending=True)
colors_br = [PALETTE_ANOM if l==1 else PALETTE_NORMAL for l in br["true_label"]]
ax5.barh(br["batch_id"].astype(str), br["max_score"],
         color=colors_br, alpha=0.85, edgecolor="none")
ax5.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5)
ax5.set_xlabel("Max Anomali Skoru")
ax5.set_title("Parti Bazında Max Skor")
ax5.legend(handles=[
    mpatches.Patch(color=PALETTE_NORMAL, label="Normal"),
    mpatches.Patch(color=PALETTE_ANOM,   label="Anormal"),
    Line2D([0],[0], color=PALETTE_THRESH, ls="--", lw=1.5,
           label=f"Eşik={BEST_THRESHOLD:.4f}"),
], fontsize=7)

# 11f: Skor dağılımı
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(meta_test[meta_test["label"]==0]["anomaly_score"], bins=60,
         color=PALETTE_NORMAL, alpha=0.6, density=True,
         label="Normal", edgecolor="none")
ax6.hist(meta_test[meta_test["label"]==1]["anomaly_score"], bins=60,
         color=PALETTE_ANOM,   alpha=0.6, density=True,
         label="Anormal", edgecolor="none")
ax6.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5, label="Eşik")
ax6.set_xlabel("Anomali Skoru (-decision_function)")
ax6.set_ylabel("Yoğunluk")
ax6.set_title("Skor Dağılımı: Normal vs Anormal")
ax6.legend(fontsize=8)

plt.savefig("performans_metrikleri_ocsvm.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("performans_metrikleri_ocsvm.png")
print("✓ Performans metrikleri grafiği indirildi.")

print("\n" + "═"*55)
print("  PERFORMANS RAPORU — PARTİ DÜZEYİ  [OCSVM]")
print("═"*55)
print(classification_report(
    batch_results["true_label"], batch_results["pred_label"],
    target_names=["Normal","Anormal"], zero_division=0,
))
print(f"ROC-AUC        : {auc_val:.4f}")
print(f"Avg Precision  : {ap:.4f}")
print(f"Kullanılan Eşik: {BEST_THRESHOLD:.4f}")


### HÜCRE 12: Anomali Skor Zaman Serileri

In [ ]:
# ── HÜCRE 12: Anomali Skor Zaman Serileri ───────────────────────────────────

batches_sorted = (
    batch_results.sort_values(["true_label","batch_id"])["batch_id"].tolist()
)
n_cols = 3
n_rows = int(np.ceil(len(batches_sorted) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*4))
fig.suptitle("Anomali Skoru Zaman Serileri — OCSVM",
             fontsize=14, color="#FFFFFF")
axes = axes.flatten()

for i, bid in enumerate(batches_sorted):
    ax   = axes[i]
    bdat = meta_test[meta_test["batch_id"]==bid].sort_values("t")
    lbl  = batch_results.loc[batch_results["batch_id"]==bid,"true_label"].iloc[0]
    pred = batch_results.loc[batch_results["batch_id"]==bid,"pred_label"].iloc[0]
    color = PALETTE_ANOM if lbl==1 else PALETTE_NORMAL

    ax.fill_between(bdat["t"], bdat["anomaly_score"],
                    alpha=0.18, color=color)
    ax.plot(bdat["t"], bdat["anomaly_score"],
            color=PALETTE_SCORE, lw=1.2, alpha=0.9)
    ax.axhline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.2,
               label=f"Eşik {BEST_THRESHOLD:.3f}")

    breach = bdat[bdat["anomaly_score"] > BEST_THRESHOLD]
    if not breach.empty:
        first_t = breach["t"].iloc[0]
        ax.axvline(first_t, color="#FF9F43", ls=":", lw=1.4,
                   label=f"İlk alarm: t={first_t}")

    status = "✓" if lbl==pred else "✗"
    title_color = "#3DD68C" if lbl==pred else "#FF5370"
    ax.set_title(f"{bid}  [{'ANORMAL' if lbl==1 else 'NORMAL'}]  {status}",
                 fontsize=9, color=title_color)
    ax.set_xlabel("t (ölçüm sırası)", fontsize=7)
    ax.set_ylabel("Skor", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=6)

for j in range(len(batches_sorted), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("anomali_skor_zaman_serileri_ocsvm.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("anomali_skor_zaman_serileri_ocsvm.png")
print("✓ Anomali skor zaman serileri indirildi.")


### HÜCRE 13: Sensör Katkı Analizi — Manuel Permütasyon (IF ile aynı proxy)

In [ ]:
# ── HÜCRE 13: Sensör Katkı Analizi — Manuel Permütasyon (IF ile aynı proxy) ─
# Strateji: Her sensörün tüm pencere feature'larını sıfırla (ölçeklenmiş ortalama=0),
# baseline ile karşılaştır → skor düşüşü = o sensörün katkısı

print("Sensör katkı analizi hesaplanıyor (≈1-2 dk)...")

baseline_scores = model.decision_function(X_test_sc)  # büyük = normal

# Anormal partiler, eşik üstü noktalar
anom_mask = (
    meta_test["batch_id"].isin(anom_batch_ids) &
    (meta_test["anomaly_score"] > BEST_THRESHOLD)
).values

sensor_importance      = {}   # tüm test
sensor_importance_anom = {}   # sadece anormal+eşik üstü

for s_idx, s_name in enumerate(SENSOR_LABELS):
    X_perturbed = X_test_sc.copy()
    # Bu sensörün window içindeki tüm zaman adımlarını sıfırla
    for w in range(WINDOW_SIZE):
        feat_idx = w * len(SENSOR_COLS) + s_idx
        X_perturbed[:, feat_idx] = 0.0   # ölçeklenmiş ortalama = 0

    perturbed_scores = model.decision_function(X_perturbed)

    # Genel etki
    delta_all  = np.abs(baseline_scores - perturbed_scores).mean()
    # Anormal bölgedeki etki
    delta_anom = np.abs(
        baseline_scores[anom_mask] - perturbed_scores[anom_mask]
    ).mean() if anom_mask.sum() > 0 else 0.0

    sensor_importance[s_name]      = delta_all
    sensor_importance_anom[s_name] = delta_anom

imp_df = pd.DataFrame(
    list(sensor_importance.items()), columns=["Sensör","Önem"]
).sort_values("Önem", ascending=True)

imp_df_anom = pd.DataFrame(
    list(sensor_importance_anom.items()), columns=["Sensör","Önem"]
).sort_values("Önem", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sensör Katkı Analizi — OCSVM  (Permütasyon Proxy)",
             fontsize=12, color="#FFFFFF")

for ax, df_imp, title in zip(
    axes,
    [imp_df, imp_df_anom],
    ["Genel Sensör Önemi (Tüm Test)",
     "Anormal Partiler — Eşik Üstü Noktalar"]
):
    colors_imp = plt.cm.YlOrRd(np.linspace(0.25, 0.92, len(df_imp)))
    bars = ax.barh(df_imp["Sensör"], df_imp["Önem"],
                   color=colors_imp, alpha=0.9, edgecolor="none")
    ax.set_xlabel("Ortalama Skor Değişimi (Sensör Katkısı)")
    ax.set_title(title)
    for bar, val in zip(bars, df_imp["Önem"]):
        ax.text(bar.get_width() + max(df_imp["Önem"])*0.01,
                bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8, color="#C8CDD8")

top_sensor = imp_df_anom.iloc[-1]["Sensör"]   # en yüksek önem
plt.tight_layout()
plt.savefig("sensor_katkisi_ocsvm.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("sensor_katkisi_ocsvm.png")
print(f"✓ Sensör katkı analizi indirildi.")
print(f"\n  → Anomali sırasında en etkili sensör: [{top_sensor}]")


### HÜCRE 14: Özet Tablo & Tüm Dosyaları İndir

In [ ]:
# ── HÜCRE 14: Özet Tablo & Tüm Dosyaları İndir ──────────────────────────────

summary_df = batch_results.copy()
summary_df["Durum"]    = summary_df["true_label"].map({0:"Normal",1:"Anormal"})
summary_df["Tahmin"]   = summary_df["pred_label"].map({0:"Normal",1:"Anormal"})
summary_df["Doğru_mu"] = summary_df["true_label"] == summary_df["pred_label"]
summary_df["İlk_Alarm_t"] = summary_df["batch_id"].apply(
    lambda bid: (
        int(meta_test.loc[
            (meta_test["batch_id"]==bid) &
            (meta_test["anomaly_score"] > BEST_THRESHOLD), "t"
        ].min())
        if not meta_test.loc[
            (meta_test["batch_id"]==bid) &
            (meta_test["anomaly_score"] > BEST_THRESHOLD), "t"
        ].empty
        else None
    )
)
summary_df = summary_df[[
    "batch_id","Durum","Tahmin","Doğru_mu",
    "max_score","mean_score","İlk_Alarm_t"
]].rename(columns={
    "batch_id":"Parti","max_score":"Max_Skor","mean_score":"Ort_Skor"
})

print("═"*55)
print("  PARTİ BAZINDA SONUÇ ÖZETİ  [OCSVM]")
print("═"*55)
print(summary_df.to_string(index=False))

# Dosyaları kaydet & indir
meta_test.to_csv("anomali_skorlari_tam_ocsvm.csv",
                 index=False, encoding="utf-8-sig")
summary_df.to_csv("parti_sonuc_ozeti_ocsvm.csv",
                  index=False, encoding="utf-8-sig")

model_params = {
    "model"          : "One-Class SVM",
    "kernel"         : "rbf",
    "nu"             : 0.05,
    "gamma"          : "scale",
    "window_size"    : WINDOW_SIZE,
    "n_features"     : WINDOW_SIZE * len(SENSOR_COLS),
    "anomaly_score"  : "-decision_function (yüksek = anomali)",
    "best_threshold" : round(float(BEST_THRESHOLD), 6),
    "roc_auc"        : round(float(auc_val), 4),
    "avg_precision"  : round(float(ap), 4),
    "n_support_vectors": int(n_sv),
    "n_train_samples": int(len(X_train)),
    "n_sensors"      : len(SENSOR_COLS),
    "sensor_names"   : SENSOR_LABELS,
    "top_sensor_anom": top_sensor,
}
with open("model_parametreleri_ocsvm.json", "w", encoding="utf-8") as f:
    json.dump(model_params, f, ensure_ascii=False, indent=2)

files.download("anomali_skorlari_tam_ocsvm.csv")
files.download("parti_sonuc_ozeti_ocsvm.csv")
files.download("model_parametreleri_ocsvm.json")
print("\n✓ Tüm CSV ve JSON dosyaları indirildi.")

print("\n" + "═"*58)
print("  TÜM ADIMLAR TAMAMLANDI ✓  [One-Class SVM]")
print("═"*58)
print("  İndirilen dosyalar:")
print("    📊 eda_genel_bakis_ocsvm.png")
print("    📈 eda_zaman_serileri_ocsvm.png")
print("    📉 eda_kde_dagilimlar_ocsvm.png")
print("    🎯 performans_metrikleri_ocsvm.png")
print("    ⏱  anomali_skor_zaman_serileri_ocsvm.png")
print("    🔍 sensor_katkisi_ocsvm.png")
print("    💾 anomali_skorlari_tam_ocsvm.csv")
print("    📋 parti_sonuc_ozeti_ocsvm.csv")
print("    ⚙️  model_parametreleri_ocsvm.json")
